#  데이터 수집 및 변수 생성 파이프라인

- 아파트 실거래 데이터 + 지하철역 + 학교 + 금리 → 통합 데이터셋 생성
- 카카오 API 지오코딩 → 거리 변수 생성

## 1. 카카오 REST API 키 테스트

본인 카카오 REST API 키가 정상 작동하는지 확인

In [ ]:
import requests
KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"


url = "https://dapi.kakao.com/v2/local/search/address.json"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
params = {"query": "서울특별시 강남구 광평로19길 15"}

response = requests.get(url, headers=headers, params=params)

print("상태 코드:", response.status_code)
print("응답:", response.json())

상태 코드: 200
응답: {'documents': [{'address': {'address_name': '서울 강남구 일원동 716', 'b_code': '1168011400', 'h_code': '1168072000', 'main_address_no': '716', 'mountain_yn': 'N', 'region_1depth_name': '서울', 'region_2depth_name': '강남구', 'region_3depth_h_name': '일원본동', 'region_3depth_name': '일원동', 'sub_address_no': '', 'x': '127.084797907539', 'y': '37.485530536641'}, 'address_name': '서울 강남구 광평로19길 15', 'address_type': 'ROAD_ADDR', 'road_address': {'address_name': '서울 강남구 광평로19길 15', 'building_name': '목련타운아파트', 'main_building_no': '15', 'region_1depth_name': '서울', 'region_2depth_name': '강남구', 'region_3depth_name': '일원동', 'road_name': '광평로19길', 'sub_building_no': '', 'underground_yn': 'N', 'x': '127.084797907539', 'y': '37.485530536641', 'zone_no': '06355'}, 'x': '127.084797907539', 'y': '37.485530536641'}], 'meta': {'is_end': True, 'pageable_count': 1, 'total_count': 1}}


### 결과 확인

- **200**: 정상 작동
- **401**: 키 오류 → 카카오 콘솔에서 키 다시 확인
- **403**: 권한 없음 → 카카오맵 로컬 API 활성화 필요
- **429**: 호출 한도 초과

In [2]:
if response.status_code == 200 and response.json().get('documents'):
    doc = response.json()['documents'][0]
    print("✅ API 정상 작동")
    print(f"주소: {doc['address_name']}")
    print(f"경도(x): {doc['x']}")
    print(f"위도(y): {doc['y']}")
else:
    print("❌ 확인 필요")

✅ API 정상 작동
주소: 서울 강남구 광평로19길 15
경도(x): 127.084797907539
위도(y): 37.485530536641


## 2. 실거래 데이터 통합 + 서울 필터링

In [5]:
import pandas as pd

files = [
    '아파트(매매)_실거래가_20260507111832.csv',
    '아파트(매매)_실거래가_20260507111846.csv',
    '아파트(매매)_실거래가_20260507111859.csv',
]

df_list = [pd.read_csv(f, encoding='cp949', skiprows=15) for f in files]
df = pd.concat(df_list, ignore_index=True)

df = df[df['시군구'].str.startswith('서울특별시')].reset_index(drop=True)

print('서울 거래 건수:', len(df))
df.head(3)

서울 거래 건수: 106124


,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),동,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자
0,1,서울특별시 강남구 일원동,716,716,0,목련타운,99.79,202301,1,"175,000",-,6,-,-,1993,광평로19길 15,20230508,중개거래,서울 강남구,-
1,2,서울특별시 강남구 일원동,716,716,0,목련타운,99.79,202301,1,"175,000",109,6,-,-,1993,광평로19길 15,-,중개거래,서울 강남구,23.05.11
2,3,서울특별시 서초구 반포동,18-1,18,1,래미안퍼스티지,84.85,202301,1,"300,000",103,19,-,-,2009,반포대로 275,-,중개거래,서울 서초구,23.03.30


## 3. 단지 unique 추출 + 도로명 전체 주소 조립

- 같은 단지는 한 번만 지오코딩하기 위함
- `시군구`에서 시/도 + 구까지만 가져와서 `도로명`과 결합

In [6]:
df['시구'] = df['시군구'].str.split().str[:2].str.join(' ')
df['도로명_전체'] = df['시구'] + ' ' + df['도로명']

단지_master = df[['단지명', '도로명_전체']].drop_duplicates().reset_index(drop=True)

print('unique 단지 수:', len(단지_master))
단지_master.head()

unique 단지 수: 6690


,단지명,도로명_전체
0,목련타운,서울특별시 강남구 광평로19길 15
1,래미안퍼스티지,서울특별시 서초구 반포대로 275
2,래미안 허브리츠,서울특별시 동대문구 정릉천동로 36
3,대림e-편한세상,서울특별시 동대문구 한천로63길 10
4,상도역롯데캐슬파크엘,서울특별시 동작구 양녕로 220


## 4. 카카오 지오코딩

- 단지 마스터의 `도로명_전체`를 위경도로 변환
- 예상 소요 시간: 단지 수 × 약 0.2초 (수천 건이면 10~30분)

In [15]:
import requests
from tqdm import tqdm
import time

def geocode(address):
    url = 'https://api.vworld.kr/req/address'
    params = {
        'service': 'address',
        'request': 'getCoord',
        'crs': 'epsg:4326',
        'type': 'ROAD',
        'address': address,
        'key': 'YOUR_VWORLD_API_KEY',
        'format': 'json',
    }
    try:
        r = requests.get(url, params=params, timeout=5)
        data = r.json()
        if data['response']['status'] == 'OK':
            point = data['response']['result']['point']
            return float(point['y']), float(point['x'])
    except requests.exceptions.RequestException as e:
        print(f'네트워크 오류: {address} → {e}')
    except (KeyError, ValueError):
        pass
    return None, None

lat_list, lon_list = [], []
for addr in tqdm(단지_master['도로명_전체']):
    lat, lon = geocode(addr)
    lat_list.append(lat)
    lon_list.append(lon)
    time.sleep(0.1)

단지_master['위도'] = lat_list
단지_master['경도'] = lon_list

성공 = 단지_master['위도'].notna().sum()
print(f'지오코딩 성공: {성공} / {len(단지_master)} ({성공/len(단지_master)*100:.1f}%)')
단지_master.head()

 29%|██▊       | 1909/6690 [05:42<2:11:58,  1.66s/it]

네트워크 오류: 서울특별시 성동구 독서당로 155 → HTTPSConnectionPool(host='api.vworld.kr', port=443): Read timed out. (read timeout=5)


100%|██████████| 6690/6690 [19:26<00:00,  5.74it/s]  

지오코딩 성공: 6670 / 6690 (99.7%)


,단지명,도로명_전체,위도,경도
0,목련타운,서울특별시 강남구 광평로19길 15,37.485635,127.085466
1,래미안퍼스티지,서울특별시 서초구 반포대로 275,37.502392,126.995939
2,래미안 허브리츠,서울특별시 동대문구 정릉천동로 36,37.575146,127.036668
3,대림e-편한세상,서울특별시 동대문구 한천로63길 10,37.604270,127.063500
4,상도역롯데캐슬파크엘,서울특별시 동작구 양녕로 220,37.501453,126.946230


In [16]:
실패 = 단지_master[단지_master['위도'].isna()]
print(실패[['단지명', '도로명_전체']].to_string())

            단지명                   도로명_전체
112        서희융창   서울특별시 서초구 남부순환로289길 16
553       삼성래미안     서울특별시 강남구 남부순환로 2803
1277       유원제일    서울특별시 영등포구 국회대로29길 13
1908     모닝빌아파트       서울특별시 성동구 독서당로 155
1962   스위트드림아파트    서울특별시 강서구 화곡로21길 71-1
2048         진명     서울특별시 양천구 신목로5길 23-6
2103       홍익한신        서울특별시 성동구 마장로 219
2242         삼익      서울특별시 서초구 효령로34길 79
2315       진주맨션      서울특별시 영등포구 선유로9길 31
2650        은하수      서울특별시 강남구 언주로70길 24
2864      명수대한양      서울특별시 동작구 현충로10길 42
3141        신장미     서울특별시 성동구 왕십리로 66-15
3223         공성     서울특별시 동대문구 정릉천동로 100
3731    북한산포레스트      서울특별시 은평구 서오릉로 80-1
3780      노량진맨션    서울특별시 동작구 장승배기로18길 27
4811         대명   서울특별시 강동구 고덕로25길 13-11
4822      모닝아트빌      서울특별시 서초구 효령로21길 47
4833  현대(220-2)       서울특별시 금천구 탑골로3길 50
5072         혜성      서울특별시 마포구 마포대로19길 8
5305         은성  서울특별시 동작구 장승배기로18길 20-3


### 🔍 VWorld API 테스트

In [17]:
import requests

VWORLD_KEY = "YOUR_VWORLD_API_KEY"

sample_addr = '서울특별시 강남구 광평로19길 15'

url = 'https://api.vworld.kr/req/address'
params = {
    'service': 'address',
    'request': 'getCoord',
    'crs': 'epsg:4326',
    'type': 'ROAD',
    'address': sample_addr,
    'key': VWORLD_KEY,
    'format': 'json',
}
r = requests.get(url, params=params, timeout=5)

print('상태코드:', r.status_code)
print('응답:', r.json())

상태코드: 200
응답: {'response': {'service': {'name': 'address', 'version': '2.0', 'operation': 'getCoord', 'time': '18(ms)'}, 'status': 'OK', 'input': {'type': 'ROAD', 'address': '서울특별시 강남구 광평로19길 15'}, 'refined': {'text': '서울특별시 강남구 광평로19길 15 (일원동)', 'structure': {'level0': '대한민국', 'level1': '서울특별시', 'level2': '강남구', 'level3': '일원동', 'level4L': '광평로19길', 'level4LC': '', 'level4A': '일원본동', 'level4AC': '1168072000', 'level5': '15', 'detail': '목련타운 유치원동'}}, 'result': {'crs': 'EPSG:4326', 'point': {'x': '127.08546593121802', 'y': '37.485634708242436'}}}}


In [18]:
단지_master.to_csv('단지_master_좌표.csv', index=False, encoding='utf-8-sig')
print('중간 저장 완료: 단지_master_좌표.csv')

중간 저장 완료: 단지_master_좌표.csv


## 5. 지하철역 / 학교 데이터 로드

In [19]:
역 = pd.read_csv('서울시 역사마스터 정보.csv', encoding='cp949')
학교 = pd.read_csv('한국교육시설안전원_초중등학교위치_20260320.csv', encoding='utf-8')

학교 = 학교[학교['소재지지번주소'].str.startswith('서울', na=False)].reset_index(drop=True)

print('역:', len(역), '개')
print('서울 학교:', len(학교), '개')
print(학교['학교급구분'].value_counts())

역: 783 개
서울 학교: 1313 개
학교급구분
초등학교    606
중학교     388
고등학교    319
Name: count, dtype: int64


## 6. 거리 변수 계산

- haversine 공식으로 단지-역, 단지-학교 거리 계산
- 최근접 역명/거리, 최근접 학교명/거리, 반경 1km 내 학교 수

In [20]:
import numpy as np

def haversine_array(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def nearest(target_lat, target_lon, ref_df, lat_col='위도', lon_col='경도', name_col='역사명', extra_col=None):
    dist = haversine_array(target_lat, target_lon, ref_df[lat_col].values, ref_df[lon_col].values)
    idx = np.argmin(dist)
    result = {'name': ref_df[name_col].iloc[idx], 'dist': dist[idx]}
    if extra_col:
        result['extra'] = ref_df[extra_col].iloc[idx]
    return result

def count_within(target_lat, target_lon, ref_df, radius=1000, lat_col='위도', lon_col='경도'):
    dist = haversine_array(target_lat, target_lon, ref_df[lat_col].values, ref_df[lon_col].values)
    return int((dist <= radius).sum())

In [21]:
초 = 학교[학교['학교급구분'] == '초등학교'].reset_index(drop=True)
중 = 학교[학교['학교급구분'] == '중학교'].reset_index(drop=True)
고 = 학교[학교['학교급구분'] == '고등학교'].reset_index(drop=True)

results = {
    '최근접_역명': [], '최근접_역_호선': [], '최근접_역_거리(m)': [],
    '최근접_초등학교': [], '최근접_초_거리(m)': [], '초등학교_수': [],
    '최근접_중학교': [], '최근접_중_거리(m)': [], '중학교_수': [],
    '최근접_고등학교': [], '최근접_고_거리(m)': [], '고등학교_수': [],
}

for _, row in tqdm(단지_master.iterrows(), total=len(단지_master)):
    lat, lon = row['위도'], row['경도']
    if pd.isna(lat) or pd.isna(lon):
        for k in results:
            results[k].append(None)
        continue
    
    r = nearest(lat, lon, 역, name_col='역사명', extra_col='호선')
    results['최근접_역명'].append(r['name'])
    results['최근접_역_호선'].append(r['extra'])
    results['최근접_역_거리(m)'].append(round(r['dist'], 1))
    
    for label, ref in [('초', 초), ('중', 중), ('고', 고)]:
        r = nearest(lat, lon, ref, name_col='학교명')
        col_name = {'초': '초등학교', '중': '중학교', '고': '고등학교'}[label]
        results[f'최근접_{col_name}'].append(r['name'])
        results[f'최근접_{label}_거리(m)'].append(round(r['dist'], 1))
        results[f'{col_name}_수'].append(count_within(lat, lon, ref))

for k, v in results.items():
    단지_master[k] = v

단지_master.head()

100%|██████████| 6690/6690 [00:01<00:00, 5853.15it/s]


,단지명,도로명_전체,위도,경도,최근접_역명,최근접_역_호선,최근접_역_거리(m),최근접_초등학교,최근접_초_거리(m),초등학교_수,최근접_중학교,최근접_중_거리(m),중학교_수,최근접_고등학교,최근접_고_거리(m),고등학교_수
0,목련타운,서울특별시 강남구 광평로19길 15,37.485635,127.085466,일원,3호선,237.1,서울왕북초등학교,93.0,5.0,대왕중학교,292.8,2.0,중산고등학교,356.4,3.0
1,래미안퍼스티지,서울특별시 서초구 반포대로 275,37.502392,126.995939,신반포,9호선,113.7,서울잠원초등학교,199.4,3.0,세화여자중학교,157.5,3.0,세화여자고등학교,157.0,2.0
2,래미안 허브리츠,서울특별시 동대문구 정릉천동로 36,37.575146,127.036668,용두(동대문구청),2호선,176.5,서울신답초등학교,617.5,5.0,숭인중학교,668.0,3.0,대광고등학교,1047.4,0.0
3,대림e-편한세상,서울특별시 동대문구 한천로63길 10,37.604270,127.063500,신이문,경원선,431.0,서울이문초등학교,290.7,3.0,석관중학교,515.1,1.0,석관고등학교,655.9,1.0
4,상도역롯데캐슬파크엘,서울특별시 동작구 양녕로 220,37.501453,126.946230,상도,7호선,213.5,서울신상도초등학교,188.0,4.0,장승중학교,512.0,3.0,구암고등학교,1045.8,0.0


## 7. 원본 데이터에 join

In [22]:
df_final = df.merge(단지_master, on=['단지명', '도로명_전체'], how='left')
df_final = df_final.drop(columns=['시구'])

print('최종 행 수:', len(df_final))
print('최종 컬럼 수:', len(df_final.columns))
df_final.head(3)

최종 행 수: 106124
최종 컬럼 수: 35


,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),...,최근접_역_거리(m),최근접_초등학교,최근접_초_거리(m),초등학교_수,최근접_중학교,최근접_중_거리(m),중학교_수,최근접_고등학교,최근접_고_거리(m),고등학교_수
0,1,서울특별시 강남구 일원동,716,716,0,목련타운,99.79,202301,1,"175,000",...,237.1,서울왕북초등학교,93.0,5.0,대왕중학교,292.8,2.0,중산고등학교,356.4,3.0
1,2,서울특별시 강남구 일원동,716,716,0,목련타운,99.79,202301,1,"175,000",...,237.1,서울왕북초등학교,93.0,5.0,대왕중학교,292.8,2.0,중산고등학교,356.4,3.0
2,3,서울특별시 서초구 반포동,18-1,18,1,래미안퍼스티지,84.85,202301,1,"300,000",...,113.7,서울잠원초등학교,199.4,3.0,세화여자중학교,157.5,3.0,세화여자고등학교,157.0,2.0


## 8. 기준금리 컬럼 추가

- 한국은행 기준금리 데이터를 wide → long 변환
- 계약년월 기준으로 매핑하여 `기준금리` 컬럼 추가

In [24]:
rate = pd.read_csv('한국은행 기준금리 및 여수신금리_10021335.csv', encoding='utf-8-sig')

rate_long = rate.melt(
    id_vars=['통계표', '계정항목', '단위', '변환'],
    var_name='연월',
    value_name='기준금리'
)

rate_long['계약년월'] = rate_long['연월'].str.replace('/', '').astype(int)
rate_long = rate_long[['계약년월', '기준금리']].reset_index(drop=True)

print(f'총 {len(rate_long)}개월')
rate_long.head()

총 36개월


,계약년월,기준금리
0,202201,1.25
1,202202,1.25
2,202203,1.25
3,202204,1.50
4,202205,1.75


In [25]:
df_final['기준금리'] = df_final['계약년월'].map(
    rate_long.set_index('계약년월')['기준금리']
)

print('기준금리 결측:', df_final['기준금리'].isna().sum())
print('컬럼 수:', len(df_final.columns))
df_final[['계약년월', '기준금리']].head()

기준금리 결측: 0
컬럼 수: 36


,계약년월,기준금리
0,202301,3.50
1,202301,3.50
2,202301,3.50
3,202212,3.25
4,202212,3.25


## 9. 최종 CSV 저장

In [26]:
df_final.to_csv('apt_with_features.csv', index=False, encoding='utf-8-sig')
print('저장 완료: apt_with_features.csv')

저장 완료: apt_with_features.csv
